# Scale factor experiment: farmer-network scaling

This notebook analyzes the `scale_n_farmers` experiment. The experimental unit is the instance seed `n_id`; within each seed, the script sweeps `scale_factor`. Therefore, the main plots use **within-seed paired changes** relative to `scale_factor = 1.0`.

Interpretation caveat: this is a common-random-seed farmer-network scaling experiment, not a fixed-realized-farmer counterfactual. The generator changes the number of farmer points, and farmer quantities are rescaled to satisfy intermediary capacity constraints. So `scale_factor` should be interpreted as changing farmer-network density / fragmentation, not necessarily proportional total supply.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ============================================================
# Config
# ============================================================

RESULTS_DIR = Path("../results/scale_n_farmers")
PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_SCALE_FACTOR = 1.0
USD = 1

EXPECTED_SCALE_FACTORS = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]
ALPHA = 0.05


# ============================================================
# Load raw JSON files
# ============================================================

def load_raw_results(results_dir: Path):
    """
    Load all JSON files in results_dir.

    Expected file structure:
        results/scale_n_farmers/{n_id}.json

    Each file contains a list of results, one per scale_factor value.
    """
    all_results = []
    json_paths = sorted(results_dir.glob("*.json"))

    if not json_paths:
        raise FileNotFoundError(f"No JSON files found in {results_dir}")

    for path in json_paths:
        with open(path, "r") as f:
            results = json.load(f)

        for result in results:
            result["_source_file"] = path.name
            all_results.append(result)

    return all_results


# ============================================================
# Process one raw result into one clean row
# ============================================================

def safe_get(dct, keys, default=np.nan):
    """Nested dictionary getter."""
    cur = dct
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def sum_values(dct):
    if not isinstance(dct, dict):
        return np.nan
    return np.sum([float(v) for v in dct.values()])


def expected_matched_count(dct):
    """
    Matched intermediaries are stored as matching probabilities.
    Use the expected matched count, not bool(v), which would count any
    positive fractional probability as one full matched intermediary.
    """
    if not isinstance(dct, dict):
        return np.nan
    return np.sum([float(v) for v in dct.values()])


def build_scale_row(result, usd=USD):
    """
    Convert one raw simulation result into a compact analysis row.

    Economic metrics are normalized by total fruit value:
        total fruit value = sum farmer quantities * fruit price
    """
    sv = result["summary_vanilla"]

    max_sol = sv["max_int_welf_sol"]      # intermediary-favorable tie-break
    min_sol = sv["min_int_welf_sol"]      # farmer-favorable tie-break

    farmer_quantities = result["farmer_quantities"]
    total_quantity = sum_values(farmer_quantities)

    fruit_price = sv["instance"]["fruit_price"]
    fruit_value = total_quantity * fruit_price
    if fruit_value == 0 or pd.isna(fruit_value):
        fruit_value = np.nan

    matched_max = max_sol.get("matched_intermediaries", {})
    matched_min = min_sol.get("matched_intermediaries", {})

    n_ints = int(result["n_ints"])
    n_farmers = safe_get(sv, ["instance", "n_farmers"])
    n_intermediaries = safe_get(sv, ["instance", "n_intermediaries"])
    scale_factor = float(result["scale_factor"])

    row = {
        # Identifiers
        "source_file": result.get("_source_file"),
        "instance_id": result["instance_id"],
        "n_id": result["n_id"],
        "n_ints": n_ints,
        "scale_factor": scale_factor,

        # Instance scale
        "n_farmers": n_farmers,
        "n_intermediaries": n_intermediaries,
        "total_quantity": total_quantity,
        "avg_quantity_per_farmer": total_quantity / n_farmers if pd.notna(n_farmers) and n_farmers > 0 else np.nan,
        "fruit_price": fruit_price,
        "fruit_value": fruit_value / usd,

        # Raw economic values for the intermediary-favorable solution
        "profit_raw": max_sol["profit"] / usd,
        "farmer_welfare_raw": max_sol["farmer_welfare"] / usd,
        "intermediary_welfare_raw": max_sol["intermediary_welfare"] / usd,
        "total_welfare_raw": (max_sol["farmer_welfare"] + max_sol["intermediary_welfare"]) / usd,
        "matching_cost_raw": max_sol["matching_cost"] / usd,

        # Intermediary-favorable solution shares
        "profit_share": max_sol["profit"] / fruit_value,
        "farmer_welfare_share": max_sol["farmer_welfare"] / fruit_value,
        "intermediary_welfare_share": max_sol["intermediary_welfare"] / fruit_value,
        "total_welfare_share": (max_sol["farmer_welfare"] + max_sol["intermediary_welfare"]) / fruit_value,
        "matching_cost_share": max_sol["matching_cost"] / fruit_value,

        # Farmer-favorable solution shares
        "farmer_fav_profit_share": min_sol["profit"] / fruit_value,
        "farmer_fav_farmer_welfare_share": min_sol["farmer_welfare"] / fruit_value,
        "farmer_fav_intermediary_welfare_share": min_sol["intermediary_welfare"] / fruit_value,
        "farmer_fav_total_welfare_share": (min_sol["farmer_welfare"] + min_sol["intermediary_welfare"]) / fruit_value,
        "farmer_fav_matching_cost_share": min_sol["matching_cost"] / fruit_value,

        # Optional forced/suboptimal quantities
        "forced_lower_bound_share": sv["forced_lower_bound"] / fruit_value if sv.get("forced_lower_bound") is not None else np.nan,
        "forced_upper_bound_share": sv["forced_upper_bound"] / fruit_value if sv.get("forced_upper_bound") is not None else np.nan,
        "forced_cost_share": sv["forced_cost"] / fruit_value if sv.get("forced_cost") is not None else np.nan,

        # Price coefficients
        "price_per_quantity": max_sol.get("price_per_quantity", np.nan),
        "price_per_mile_paved": max_sol.get("price_per_mile_paved", np.nan),
        "price_per_mile_dirt": max_sol.get("price_per_mile_dirt", np.nan),

        # Runtime / solver metrics
        "time_vanilla": sv["total_time"],
        "oracle_calls": sv["total_oracle_calls"],

        # Matching saturation using expected matched count
        "n_matched_intermediaries": expected_matched_count(matched_max),
        "matched_share": expected_matched_count(matched_max) / n_ints,
        "farmer_fav_n_matched_intermediaries": expected_matched_count(matched_min),
        "farmer_fav_matched_share": expected_matched_count(matched_min) / n_ints,

        # Epsilon/cost summaries
        "avg_epsilon": np.mean(list(result["epsilon"].values())),
        "avg_cost": np.mean(list(result["cost"].values())),
    }

    # Computational metrics normalized by problem size
    if pd.notna(n_farmers) and n_farmers != 0:
        row["time_per_farmer"] = row["time_vanilla"] / n_farmers
        row["oracle_calls_per_farmer"] = row["oracle_calls"] / n_farmers
    else:
        row["time_per_farmer"] = np.nan
        row["oracle_calls_per_farmer"] = np.nan

    if pd.notna(n_farmers) and n_farmers != 0 and n_ints != 0:
        row["time_per_farmer_int_pair"] = row["time_vanilla"] / (n_farmers * n_ints)
        row["oracle_calls_per_farmer_int_pair"] = row["oracle_calls"] / (n_farmers * n_ints)
    else:
        row["time_per_farmer_int_pair"] = np.nan
        row["oracle_calls_per_farmer_int_pair"] = np.nan

    return row


def build_analysis_df(results_dir=RESULTS_DIR):
    raw_results = load_raw_results(results_dir)
    rows = [build_scale_row(result) for result in raw_results]
    df = pd.DataFrame(rows)

    df["n_id"] = df["n_id"].astype(str)
    df["n_ints"] = pd.to_numeric(df["n_ints"], errors="coerce")
    df["scale_factor"] = pd.to_numeric(df["scale_factor"], errors="coerce")

    df = df.sort_values(["n_id", "scale_factor"]).reset_index(drop=True)
    return df


df = build_analysis_df(RESULTS_DIR)

print(df.shape)
display(df.head())

(1358, 44)


,source_file,instance_id,n_id,n_ints,scale_factor,n_farmers,n_intermediaries,total_quantity,avg_quantity_per_farmer,fruit_price,...,n_matched_intermediaries,matched_share,farmer_fav_n_matched_intermediaries,farmer_fav_matched_share,avg_epsilon,avg_cost,time_per_farmer,oracle_calls_per_farmer,time_per_farmer_int_pair,oracle_calls_per_farmer_int_pair
0,0.json,0_0.7,0,12,0.7,21,11,34.0,1.619048,2513000.0,...,4.0,0.333333,4.0,0.333333,2.0,430350.013604,0.312925,0.047619,0.026077,0.003968
1,0.json,0_0.8,0,12,0.8,24,12,40.6,1.691667,2513000.0,...,5.0,0.416667,5.0,0.416667,2.0,459597.424482,0.410173,0.041667,0.034181,0.003472
2,0.json,0_0.9,0,12,0.9,26,12,46.9,1.803846,2513000.0,...,6.0,0.500000,6.0,0.500000,2.0,459597.424482,0.796882,0.076923,0.066407,0.006410
3,0.json,0_1,0,12,1.0,28,12,47.3,1.689286,2513000.0,...,6.0,0.500000,6.0,0.500000,2.0,482573.608183,0.811536,0.071429,0.067628,0.005952
4,0.json,0_1.1,0,12,1.1,31,12,47.7,1.538710,2513000.0,...,6.0,0.500000,6.0,0.500000,2.0,482573.608183,0.901281,0.064516,0.075107,0.005376


## Validate the paired design

The CI calculation treats `n_id` as the IID unit and the scale-factor sweep as repeated measurements within each seed. That is appropriate for estimating the **mean paired change**. It does not describe the full cross-seed heterogeneity, so later cells also produce boxplots for selected metrics.

In [2]:
def check_complete_sweep(df, id_col="n_id", x_col="scale_factor", expected_values=EXPECTED_SCALE_FACTORS):
    observed = (
        df.groupby(id_col)[x_col]
        .apply(lambda s: sorted(np.round(pd.to_numeric(s, errors="coerce").dropna().unique(), 10)))
    )
    expected = sorted(np.round(np.array(expected_values, dtype=float), 10))
    complete = observed.apply(lambda vals: vals == expected)

    print(f"Total seeds: {len(observed)}")
    print(f"Complete seeds: {complete.sum()}")
    print(f"Incomplete seeds: {(~complete).sum()}")

    if (~complete).any():
        display(observed.loc[~complete].head(20).to_frame("observed_values"))


check_complete_sweep(df)

scale_summary = (
    df.groupby("scale_factor")
    .agg(
        n=("n_id", "count"),
        n_farmers_mean=("n_farmers", "mean"),
        n_farmers_sd=("n_farmers", "std"),
        total_quantity_mean=("total_quantity", "mean"),
        avg_quantity_per_farmer_mean=("avg_quantity_per_farmer", "mean"),
        fruit_value_mean=("fruit_value", "mean"),
        matched_count_mean=("n_matched_intermediaries", "mean"),
    )
)

display(scale_summary)

Total seeds: 194
Complete seeds: 194
Incomplete seeds: 0


,n,n_farmers_mean,n_farmers_sd,total_quantity_mean,avg_quantity_per_farmer_mean,fruit_value_mean,matched_count_mean
scale_factor,,,,,,,
0.7,194,17.515464,2.689891,30.188660,1.735210,7.586410e+07,3.876289
0.8,194,20.005155,2.956502,33.741237,1.696991,8.479173e+07,4.268041
0.9,194,22.355670,3.230729,36.987629,1.663699,9.294991e+07,4.567010
1.0,194,24.943299,3.232266,40.497938,1.632004,1.017713e+08,4.979381
1.1,194,27.453608,3.808963,43.976289,1.610551,1.105124e+08,5.386598
1.2,194,29.886598,4.205539,47.250515,1.588689,1.187405e+08,5.762887
1.3,194,32.335052,4.322026,50.419072,1.565672,1.267031e+08,6.128866


In [3]:
def add_baseline_changes(df, metrics, base_scale_factor=BASE_SCALE_FACTOR):
    """
    For each metric, compute within-seed changes relative to base_scale_factor.

    This respects the design:
        IID unit = n_id
        repeated sweep = scale_factor within n_id
    """
    out = df.copy()
    out["n_id"] = out["n_id"].astype(str)
    out["scale_factor"] = pd.to_numeric(out["scale_factor"], errors="coerce")

    for metric in metrics:
        out[metric] = pd.to_numeric(out[metric], errors="coerce")

        base = (
            out.loc[np.isclose(out["scale_factor"], base_scale_factor), ["n_id", metric]]
            .drop_duplicates(subset=["n_id"])
            .rename(columns={metric: f"{metric}_base"})
        )

        out = out.merge(base, on="n_id", how="left", validate="many_to_one")
        base_col = f"{metric}_base"

        out[f"{metric}_diff_from_base"] = out[metric] - out[base_col]
        out[f"{metric}_pct_change_from_base"] = np.where(
            out[base_col].abs() > 1e-12,
            100 * (out[metric] / out[base_col] - 1),
            np.nan,
        )

    return out


intermediary_fav_share_metrics = [
    "profit_share",
    "farmer_welfare_share",
    "intermediary_welfare_share",
    "total_welfare_share",
    "matching_cost_share",
    "matched_share",
]

farmer_fav_share_metrics = [
    "farmer_fav_profit_share",
    "farmer_fav_farmer_welfare_share",
    "farmer_fav_intermediary_welfare_share",
    "farmer_fav_total_welfare_share",
    "farmer_fav_matching_cost_share",
    "farmer_fav_matched_share",
]

computational_metrics = [
    "time_vanilla",
    "oracle_calls",
    "time_per_farmer",
    "oracle_calls_per_farmer",
    "time_per_farmer_int_pair",
    "oracle_calls_per_farmer_int_pair",
]

all_metrics = intermediary_fav_share_metrics + farmer_fav_share_metrics + computational_metrics

df = add_baseline_changes(df, all_metrics, base_scale_factor=BASE_SCALE_FACTOR)

In [4]:
def paired_change_summary(df, change_metric, alpha=ALPHA):
    """
    Compute t-based confidence intervals across IID seeds for a within-seed change metric.

    This estimates uncertainty in the mean paired change, not cross-seed heterogeneity.
    """
    plot_df = df[["n_id", "scale_factor", change_metric]].copy()

    plot_df["n_id"] = plot_df["n_id"].astype(str)
    plot_df["scale_factor"] = pd.to_numeric(plot_df["scale_factor"], errors="coerce")
    plot_df[change_metric] = pd.to_numeric(plot_df[change_metric], errors="coerce")
    plot_df = plot_df.dropna(subset=["n_id", "scale_factor", change_metric])

    summary = (
        plot_df
        .groupby("scale_factor")[change_metric]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
        .sort_values("scale_factor")
    )

    summary["se"] = summary["std"] / np.sqrt(summary["n"])
    summary["tcrit"] = summary["n"].apply(
        lambda n: stats.t.ppf(1 - alpha / 2, df=n - 1) if n > 1 else np.nan
    )
    summary["ci_low"] = summary["mean"] - summary["tcrit"] * summary["se"]
    summary["ci_high"] = summary["mean"] + summary["tcrit"] * summary["se"]

    return summary


def plot_paired_change_ci(
    df,
    change_metric,
    ylabel=None,
    title=None,
    alpha=ALPHA,
    output_dir=PLOTS_DIR,
    scale_y=1,
    filename_prefix="scale_factor",
):
    summary = paired_change_summary(df, change_metric, alpha=alpha)

    x = summary["scale_factor"].to_numpy(dtype=float)
    mean = scale_y * summary["mean"].to_numpy(dtype=float)
    ci_low = scale_y * summary["ci_low"].to_numpy(dtype=float)
    ci_high = scale_y * summary["ci_high"].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.axhline(0, linewidth=1, linestyle="--", alpha=0.6)

    ax.plot(x, mean, marker="o", linewidth=3, label="Mean paired change")
    ax.fill_between(
        x,
        ci_low,
        ci_high,
        alpha=0.25,
        label=f"{int((1 - alpha) * 100)}% CI for mean paired change",
    )

    ax.set_xlabel("Scale factor")
    ax.set_ylabel(ylabel or change_metric)
    ax.set_title(title or change_metric)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()

    safe_name = change_metric.replace(".", "_").replace("/", "_")
    out_path = output_dir / f"{filename_prefix}_{safe_name}_paired_ci.png"
    fig.savefig(out_path, dpi=300)
    plt.close(fig)

    print(f"Saved {out_path}")
    return summary


def plot_paired_change_boxplot(
    df,
    change_metric,
    ylabel=None,
    title=None,
    output_dir=PLOTS_DIR,
    scale_y=1,
    filename_prefix="scale_factor",
):
    """
    Boxplot across seed-level paired changes.

    Use this when heterogeneity is more important than uncertainty in the mean,
    or when the metric is skewed/discrete, as with oracle calls.
    """
    plot_df = df[["n_id", "scale_factor", change_metric]].copy()
    plot_df["scale_factor"] = pd.to_numeric(plot_df["scale_factor"], errors="coerce")
    plot_df[change_metric] = pd.to_numeric(plot_df[change_metric], errors="coerce")
    plot_df = plot_df.dropna(subset=["scale_factor", change_metric])
    plot_df["scaled_change"] = scale_y * plot_df[change_metric]

    x_values = sorted(plot_df["scale_factor"].unique())
    data = [plot_df.loc[np.isclose(plot_df["scale_factor"], x), "scaled_change"].to_numpy() for x in x_values]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.axhline(0, linewidth=1, linestyle="--", alpha=0.6)

    ax.boxplot(
        data,
        positions=x_values,
        widths=0.045,
        showfliers=False,
        patch_artist=False,
    )

    # Add lightly jittered seed-level points for transparency.
    rng = np.random.default_rng(0)
    for x, vals in zip(x_values, data):
        jitter = rng.normal(0, 0.006, size=len(vals))
        ax.scatter(np.full(len(vals), x) + jitter, vals, s=10, alpha=0.18)

    ax.set_xlabel("Scale factor")
    ax.set_ylabel(ylabel or change_metric)
    ax.set_title(title or change_metric)
    ax.grid(alpha=0.3)
    fig.tight_layout()

    safe_name = change_metric.replace(".", "_").replace("/", "_")
    out_path = output_dir / f"{filename_prefix}_{safe_name}_boxplot.png"
    fig.savefig(out_path, dpi=300)
    plt.close(fig)

    print(f"Saved {out_path}")

## Economic outcomes: intermediary-favorable tie-break

These are the same core economic plots as before, but with corrected labels. Share differences are shown in **percentage points**. The matched-share plot is also a percentage-point change, but its denominator is available intermediaries rather than total fruit value.

In [5]:
pretty_labels = {
    "profit_share": "Profit / total fruit value",
    "farmer_welfare_share": "Farmer welfare / total fruit value",
    "intermediary_welfare_share": "Intermediary welfare / total fruit value",
    "total_welfare_share": "Total welfare / total fruit value",
    "matching_cost_share": "Matching cost / total fruit value",
    "matched_share": "Matched intermediaries / total intermediaries",
}

for metric in intermediary_fav_share_metrics:
    change_metric = f"{metric}_diff_from_base"

    if metric == "matched_share":
        ylabel = f"Change from scale factor = {BASE_SCALE_FACTOR}\n(percentage points of available intermediaries)"
    else:
        ylabel = f"Change from scale factor = {BASE_SCALE_FACTOR}\n(percentage points of total fruit value)"

    plot_paired_change_ci(
        df,
        change_metric,
        ylabel=ylabel,
        title=f"{pretty_labels.get(metric, metric)}: paired change from baseline",
        scale_y=100,
        filename_prefix="scale_factor_intermediary_fav",
    )

Saved ../results/scale_n_farmers/plots/scale_factor_intermediary_fav_profit_share_diff_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_intermediary_fav_farmer_welfare_share_diff_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_intermediary_fav_intermediary_welfare_share_diff_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_intermediary_fav_total_welfare_share_diff_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_intermediary_fav_matching_cost_share_diff_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_intermediary_fav_matched_share_diff_from_base_paired_ci.png


## Economic outcomes: farmer-favorable tie-break

`min_int_welf_sol` is the farmer-favorable tie-break: after profit is optimized, the solver selects a solution on the profit-optimal face that is most favorable to farmers / least favorable to intermediaries. Plotting these outcomes helps check whether the patterns are robust to welfare tie-breaking.

In [6]:
farmer_fav_labels = {
    "farmer_fav_profit_share": "Farmer-favorable profit / total fruit value",
    "farmer_fav_farmer_welfare_share": "Farmer-favorable farmer welfare / total fruit value",
    "farmer_fav_intermediary_welfare_share": "Farmer-favorable intermediary welfare / total fruit value",
    "farmer_fav_total_welfare_share": "Farmer-favorable total welfare / total fruit value",
    "farmer_fav_matching_cost_share": "Farmer-favorable matching cost / total fruit value",
    "farmer_fav_matched_share": "Farmer-favorable matched intermediaries / total intermediaries",
}

for metric in farmer_fav_share_metrics:
    change_metric = f"{metric}_diff_from_base"

    if metric == "farmer_fav_matched_share":
        ylabel = f"Change from scale factor = {BASE_SCALE_FACTOR}\n(percentage points of available intermediaries)"
    else:
        ylabel = f"Change from scale factor = {BASE_SCALE_FACTOR}\n(percentage points of total fruit value)"

    plot_paired_change_ci(
        df,
        change_metric,
        ylabel=ylabel,
        title=f"{farmer_fav_labels.get(metric, metric)}: paired change from baseline",
        scale_y=100,
        filename_prefix="scale_factor_farmer_fav",
    )

Saved ../results/scale_n_farmers/plots/scale_factor_farmer_fav_farmer_fav_profit_share_diff_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_farmer_fav_farmer_fav_farmer_welfare_share_diff_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_farmer_fav_farmer_fav_intermediary_welfare_share_diff_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_farmer_fav_farmer_fav_total_welfare_share_diff_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_farmer_fav_farmer_fav_matching_cost_share_diff_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_farmer_fav_farmer_fav_matched_share_diff_from_base_paired_ci.png


## Tie-break comparison plots

These overlay the intermediary-favorable and farmer-favorable tie-breaks for the welfare outcomes. If both lines move in the same direction, the result is less likely to be just an artifact of the welfare tie-break.

In [7]:
def plot_two_tiebreaks(
    df,
    max_metric,
    farmer_fav_metric,
    *,
    title,
    ylabel,
    scale_y=100,
    output_dir=PLOTS_DIR,
):
    max_summary = paired_change_summary(df, f"{max_metric}_diff_from_base")
    fav_summary = paired_change_summary(df, f"{farmer_fav_metric}_diff_from_base")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.axhline(0, linewidth=1, linestyle="--", alpha=0.6)

    for summary, label in [
        (max_summary, "Intermediary-favorable tie-break"),
        (fav_summary, "Farmer-favorable tie-break"),
    ]:
        x = summary["scale_factor"].to_numpy(dtype=float)
        mean = scale_y * summary["mean"].to_numpy(dtype=float)
        ci_low = scale_y * summary["ci_low"].to_numpy(dtype=float)
        ci_high = scale_y * summary["ci_high"].to_numpy(dtype=float)

        ax.plot(x, mean, marker="o", linewidth=3, label=label)
        ax.fill_between(x, ci_low, ci_high, alpha=0.15)

    ax.set_xlabel("Scale factor")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()

    safe_name = title.lower().replace(" ", "_").replace("/", "_").replace(":", "")
    out_path = output_dir / f"scale_factor_{safe_name}_tiebreak_comparison.png"
    fig.savefig(out_path, dpi=300)
    plt.close(fig)
    print(f"Saved {out_path}")


plot_two_tiebreaks(
    df,
    "farmer_welfare_share",
    "farmer_fav_farmer_welfare_share",
    title="Farmer welfare: tie-break comparison",
    ylabel=f"Change from scale factor = {BASE_SCALE_FACTOR}\n(percentage points of total fruit value)",
)

plot_two_tiebreaks(
    df,
    "intermediary_welfare_share",
    "farmer_fav_intermediary_welfare_share",
    title="Intermediary welfare: tie-break comparison",
    ylabel=f"Change from scale factor = {BASE_SCALE_FACTOR}\n(percentage points of total fruit value)",
)

plot_two_tiebreaks(
    df,
    "profit_share",
    "farmer_fav_profit_share",
    title="Profit: tie-break comparison",
    ylabel=f"Change from scale factor = {BASE_SCALE_FACTOR}\n(percentage points of total fruit value)",
)

levels = (
    df.groupby("scale_factor")[[
        "farmer_welfare_share",
        "farmer_fav_farmer_welfare_share",
        "intermediary_welfare_share",
        "farmer_fav_intermediary_welfare_share",
        "profit_share",
        "farmer_fav_profit_share",
    ]]
    .mean()
    .mul(100)
)

display(levels)

Saved ../results/scale_n_farmers/plots/scale_factor_farmer_welfare_tie-break_comparison_tiebreak_comparison.png
Saved ../results/scale_n_farmers/plots/scale_factor_intermediary_welfare_tie-break_comparison_tiebreak_comparison.png
Saved ../results/scale_n_farmers/plots/scale_factor_profit_tie-break_comparison_tiebreak_comparison.png


,farmer_welfare_share,farmer_fav_farmer_welfare_share,intermediary_welfare_share,farmer_fav_intermediary_welfare_share,profit_share,farmer_fav_profit_share
scale_factor,,,,,,
0.7,85.144210,87.225205,2.140131,0.059137,4.497214,4.497214
0.8,85.600713,87.672901,2.212322,0.140134,4.020283,4.020283
0.9,86.072067,88.037464,2.182554,0.217157,3.703233,3.703233
1.0,86.458835,88.374943,2.183836,0.267728,3.310526,3.310526
1.1,86.787232,88.492797,2.301967,0.596402,2.848590,2.848590
1.2,87.180337,88.746992,2.276266,0.709611,2.451883,2.451883
1.3,87.442177,88.934675,2.299443,0.806945,2.142435,2.142435


## Computational outcomes

For runtime and oracle calls, percent-change CIs are useful for the mean paired effect. But computational metrics can be skewed and discrete, so boxplots are also included for key oracle-call metrics. The boxplots show seed-level heterogeneity rather than uncertainty in the mean.

In [8]:
pretty_compute_labels = {
    "time_vanilla": "Runtime",
    "oracle_calls": "Oracle calls",
    "time_per_farmer": "Runtime per farmer",
    "oracle_calls_per_farmer": "Oracle calls per farmer",
    "time_per_farmer_int_pair": "Runtime per farmer-intermediary pair",
    "oracle_calls_per_farmer_int_pair": "Oracle calls per farmer-intermediary pair",
}

for metric in computational_metrics:
    change_metric = f"{metric}_pct_change_from_base"

    plot_paired_change_ci(
        df,
        change_metric,
        ylabel=f"% change from scale factor = {BASE_SCALE_FACTOR}",
        title=f"{pretty_compute_labels.get(metric, metric)}: paired % change from baseline",
        scale_y=1,
        filename_prefix="scale_factor_compute",
    )

# Boxplots for computational metrics where distribution/heterogeneity matters.
for metric in [
    "oracle_calls",
    "oracle_calls_per_farmer",
    "time_vanilla",
    "time_per_farmer",
]:
    change_metric = f"{metric}_pct_change_from_base"
    plot_paired_change_boxplot(
        df,
        change_metric,
        ylabel=f"% change from scale factor = {BASE_SCALE_FACTOR}",
        title=f"{pretty_compute_labels.get(metric, metric)}: seed-level paired % changes",
        scale_y=1,
        filename_prefix="scale_factor_compute",
    )

Saved ../results/scale_n_farmers/plots/scale_factor_compute_time_vanilla_pct_change_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_compute_oracle_calls_pct_change_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_compute_time_per_farmer_pct_change_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_compute_oracle_calls_per_farmer_pct_change_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_compute_time_per_farmer_int_pair_pct_change_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_compute_oracle_calls_per_farmer_int_pair_pct_change_from_base_paired_ci.png
Saved ../results/scale_n_farmers/plots/scale_factor_compute_oracle_calls_pct_change_from_base_boxplot.png
Saved ../results/scale_n_farmers/plots/scale_factor_compute_oracle_calls_per_farmer_pct_change_from_base_boxplot.png
Saved ../results/scale_n_farmers/plots/scale_factor_compute_time_vanilla_pct_change

## Oracle-call diagnostics

The oracle-call plots can be unintuitive because oracle calls are algorithmic objects, not smooth economic primitives. They may include fixed overhead, and they depend on the structure of violated constraints, not just the number of farmers.

Since `n_ints` is fixed in this experiment, percent changes in oracle calls per farmer and per farmer-intermediary pair should have nearly the same shape; the latter just divides by a constant.

In [9]:
oracle_debug = (
    df.groupby("scale_factor")
    .agg(
        n=("n_id", "count"),
        n_farmers_mean=("n_farmers", "mean"),
        n_farmers_std=("n_farmers", "std"),
        oracle_calls_mean=("oracle_calls", "mean"),
        oracle_calls_std=("oracle_calls", "std"),
        oracle_calls_per_farmer_mean=("oracle_calls_per_farmer", "mean"),
        oracle_calls_per_farmer_int_pair_mean=("oracle_calls_per_farmer_int_pair", "mean"),
        time_mean=("time_vanilla", "mean"),
        time_per_farmer_mean=("time_per_farmer", "mean"),
    )
)

baseline = oracle_debug.loc[BASE_SCALE_FACTOR]
for col in [
    "n_farmers_mean",
    "oracle_calls_mean",
    "oracle_calls_per_farmer_mean",
    "oracle_calls_per_farmer_int_pair_mean",
    "time_mean",
    "time_per_farmer_mean",
]:
    oracle_debug[f"{col}_pct_change_from_base"] = 100 * (oracle_debug[col] / baseline[col] - 1)


display(oracle_debug)

# Scatter: actual generated farmer count versus oracle calls.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(df["n_farmers"], df["oracle_calls"], alpha=0.25, s=18)
ax.set_xlabel("Number of generated farmers")
ax.set_ylabel("Oracle calls")
ax.set_title("Oracle calls vs. generated farmer count")
ax.grid(alpha=0.3)
fig.tight_layout()
out_path = PLOTS_DIR / "scale_factor_oracle_calls_vs_n_farmers_scatter.png"
fig.savefig(out_path, dpi=300)
plt.close(fig)
print(f"Saved {out_path}")

# Binned version for readability.
df_oracle = df[["n_farmers", "oracle_calls"]].dropna().copy()
df_oracle["n_farmers_bin"] = pd.qcut(df_oracle["n_farmers"], q=8, duplicates="drop")

binned = (
    df_oracle
    .groupby("n_farmers_bin", observed=True)
    .agg(
        n_farmers_mean=("n_farmers", "mean"),
        oracle_calls_mean=("oracle_calls", "mean"),
        oracle_calls_se=("oracle_calls", lambda x: x.std(ddof=1) / np.sqrt(len(x))),
        n=("oracle_calls", "count"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(
    binned["n_farmers_mean"],
    binned["oracle_calls_mean"],
    yerr=1.96 * binned["oracle_calls_se"],
    marker="o",
    linewidth=2,
    capsize=3,
)
ax.set_xlabel("Number of generated farmers")
ax.set_ylabel("Mean oracle calls")
ax.set_title("Mean oracle calls by generated farmer count")
ax.grid(alpha=0.3)
fig.tight_layout()
out_path = PLOTS_DIR / "scale_factor_oracle_calls_by_n_farmers_binned.png"
fig.savefig(out_path, dpi=200)
plt.close(fig)
print(f"Saved {out_path}")

display(binned)

,n,n_farmers_mean,n_farmers_std,oracle_calls_mean,oracle_calls_std,oracle_calls_per_farmer_mean,oracle_calls_per_farmer_int_pair_mean,time_mean,time_per_farmer_mean,n_farmers_mean_pct_change_from_base,oracle_calls_mean_pct_change_from_base,oracle_calls_per_farmer_mean_pct_change_from_base,oracle_calls_per_farmer_int_pair_mean_pct_change_from_base,time_mean_pct_change_from_base,time_per_farmer_mean_pct_change_from_base
scale_factor,,,,,,,,,,,,,,,
0.7,194,17.515464,2.689891,1.036082,0.186978,0.060532,0.005044,3.995289,0.219307,-29.778880,-18.623482,18.013183,18.013183,-63.011337,-48.157931
0.8,194,20.005155,2.956502,1.118557,0.324103,0.056894,0.004741,5.821693,0.281653,-19.797479,-12.145749,10.920915,10.920915,-46.102360,-33.419884
0.9,194,22.355670,3.230729,1.221649,0.440614,0.055286,0.004607,8.296244,0.359973,-10.374044,-4.048583,7.785049,7.785049,-23.192792,-14.905796
1.0,194,24.943299,3.232266,1.273196,0.490958,0.051292,0.004274,10.801388,0.423029,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1.1,194,27.453608,3.808963,1.417526,0.640516,0.051880,0.004323,14.037714,0.497958,10.064063,11.336032,1.145285,1.145285,29.962131,17.712446
1.2,194,29.886598,4.205539,1.608247,0.748840,0.054138,0.004511,18.030975,0.589086,19.818144,26.315789,5.547344,5.547344,66.932027,39.254315
1.3,194,32.335052,4.322026,1.675258,0.783711,0.052101,0.004342,21.238093,0.644104,29.634222,31.578947,1.577202,1.577202,96.623749,52.260097


Saved ../results/scale_n_farmers/plots/scale_factor_oracle_calls_vs_n_farmers_scatter.png
Saved ../results/scale_n_farmers/plots/scale_factor_oracle_calls_by_n_farmers_binned.png


,n_farmers_bin,n_farmers_mean,oracle_calls_mean,oracle_calls_se,n
0,"(8.999, 18.0]",16.233010,1.048544,0.015010,206
1,"(18.0, 20.0]",19.483221,1.080537,0.022368,149
2,"(20.0, 22.0]",21.508982,1.143713,0.027227,167
3,"(22.0, 24.0]",23.496855,1.251572,0.034521,159
4,"(24.0, 27.0]",26.034934,1.362445,0.037371,229
5,"(27.0, 29.0]",28.529412,1.512605,0.061858,119
6,"(29.0, 32.0]",30.895062,1.580247,0.055885,162
7,"(32.0, 43.0]",35.419162,1.790419,0.066790,167


## Accounting checks

For each solution, normalized profit + farmer welfare + intermediary welfare + matching cost should be approximately constant and close to total fruit value share. This is a useful check against extraction or plotting bugs.

In [17]:
accounting = (
    df.groupby("scale_factor")[[
        "profit_share",
        "farmer_welfare_share",
        "intermediary_welfare_share",
        "matching_cost_share",
        "farmer_fav_profit_share",
        "farmer_fav_farmer_welfare_share",
        "farmer_fav_intermediary_welfare_share",
        "farmer_fav_matching_cost_share",
    ]]
    .mean()
)

accounting["intermediary_fav_accounting_sum"] = (
    accounting["profit_share"]
    + accounting["farmer_welfare_share"]
    + accounting["intermediary_welfare_share"]
    + accounting["matching_cost_share"]
)

accounting["farmer_fav_accounting_sum"] = (
    accounting["farmer_fav_profit_share"]
    + accounting["farmer_fav_farmer_welfare_share"]
    + accounting["farmer_fav_intermediary_welfare_share"]
    + accounting["farmer_fav_matching_cost_share"]
)

display(accounting)

,profit_share,farmer_welfare_share,intermediary_welfare_share,matching_cost_share,farmer_fav_profit_share,farmer_fav_farmer_welfare_share,farmer_fav_intermediary_welfare_share,farmer_fav_matching_cost_share,intermediary_fav_accounting_sum,farmer_fav_accounting_sum
scale_factor,,,,,,,,,,
0.7,0.044972,0.851442,0.021401,0.082184,0.044972,0.872252,0.000591,0.082184,1.0,1.0
0.8,0.040203,0.856007,0.022123,0.081667,0.040203,0.876729,0.001401,0.081667,1.0,1.0
0.9,0.037032,0.860721,0.021826,0.080421,0.037032,0.880375,0.002172,0.080421,1.0,1.0
1.0,0.033105,0.864588,0.021838,0.080468,0.033105,0.883749,0.002677,0.080468,1.0,1.0
1.1,0.028486,0.867872,0.023020,0.080622,0.028486,0.884928,0.005964,0.080622,1.0,1.0
1.2,0.024519,0.871803,0.022763,0.080915,0.024519,0.887470,0.007096,0.080915,1.0,1.0
1.3,0.021424,0.874422,0.022994,0.081159,0.021424,0.889347,0.008069,0.081159,1.0,1.0
